In [28]:
import glob
import os
from typing import List

import joblib
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pathlib import Path

In [29]:
# --- Pydantic 模型定義 API 的輸入格式 ---
# 這裡的欄位「必須」跟你訓練時的特徵完全對應
class Features(BaseModel):
    avg_temp: float
    avg_rh: float
    max_precip: float
    distance: float
    elevation_range: float
    elevation_change: float
    elevation_gain: float
    elevation_loss: float
    high_elevation: float
    max_slope_percent: float
    max_slope_degrees: float
    slope_std_dev: float
    slope_variance: float
    max_slope_lat: float
    max_slope_lon: float
    slope_neg15: float
    slope_neg15_neg10: float
    slope_neg10_neg5: float
    slope_neg5_neg1: float
    slope_neg1_1: float
    slope_1_5: float
    slope_5_10: float
    slope_10_15: float
    slope_over15: float
    accumulated_time_seconds: float
    accumulated_distance: float

In [30]:
# 取得 Notebook 目前所在目錄
notebook_path = os.getcwd()
print("目前工作目錄:", notebook_path)

# 用 pathlib 處理路徑
notebook_dir = Path(notebook_path)

# 取得上層資料夾 (從 time_prediction -> aiservices)
parent_dir = notebook_dir.parent.parent
print("上一層:", parent_dir)

# 取得 models 資料夾完整路徑
MODEL_PATH = parent_dir / "models"
print("models 路徑:", MODEL_PATH)

目前工作目錄: /home/zoe/allpass/aiservices/time_prediction
上一層: /home/zoe/allpass
models 路徑: /home/zoe/allpass/models


In [31]:
def find_latest_model_path(path: str) -> str:
    """在指定路徑中尋找最新的 .pkl 模型檔案"""
    # 列出所有 .pkl 檔案
    list_of_files = glob.glob(os.path.join(path, "time_prediction_*.pkl"))
    if not list_of_files:
        return None
    # 根據檔名 (隱含了時間戳) 找到最新的檔案
    latest_file = max(list_of_files, key=os.path.basename)
    return latest_file


latest_file = find_latest_model_path(MODEL_PATH)
print(latest_file)

/home/zoe/allpass/models/time_prediction_20250827_070415.pkl


In [32]:
def load_model():
    """在應用程式啟動時執行的函式"""
    global PIPELINE
    latest_model_path = find_latest_model_path(MODEL_PATH)

    if latest_model_path:
        print(f"Loading model from: {latest_model_path}")
        PIPELINE = joblib.load(latest_model_path)
    else:
        print(f"No model found in {MODEL_PATH}")
        # 在這裡可以決定是否要讓應用程式因找不到模型而啟動失敗
        PIPELINE = None


load_model()

Loading model from: /home/zoe/allpass/models/time_prediction_20250827_070415.pkl


In [42]:
features = Features(
    avg_temp=8.9,
    avg_rh=7,
    max_precip=0,
    distance=1713,
    elevation_range=526.2,
    elevation_change=-304.5,
    elevation_gain=19,
    elevation_loss=539.8,
    high_elevation=1,
    max_slope_percent=-74.2,
    max_slope_degrees=-36.55,
    slope_std_dev=10.57,
    slope_variance=111.64,
    max_slope_lat=24.412,
    max_slope_lon=121.309677,
    slope_neg15=67.27,
    slope_neg15_neg10=14.55,
    slope_neg10_neg5=3.64,
    slope_neg5_neg1=3.88,
    slope_neg1_1=3.64,
    slope_1_5=5.45,
    slope_5_10=1.82,
    slope_10_15=0,
    slope_over15=3,
    accumulated_time_seconds=30580,
    accumulated_distance=9813.28,
)


def predict(features: Features):
    """接收特徵並回傳預測結果"""
    if PIPELINE is None:
        raise HTTPException(status_code=503, detail="Model is not loaded")

    try:
        # 將 Pydantic 模型轉換為 DataFrame
        # input_df = pd.DataFrame([features.dict()])
        input_df = pd.DataFrame([features.model_dump()])

        # 確保 DataFrame 的欄位順序與訓練時一致
        # PIPELINE['features'] 是我們在 train.py 中刻意存下來的
        ordered_df = input_df[PIPELINE["features"]]

        # 使用儲存的 scaler 進行特徵縮放
        scaled_features = PIPELINE["scaler"].transform(ordered_df)

        # 使用集成模型進行預測
        predictions = [model.predict(scaled_features) for model in PIPELINE["models"]]

        # 計算平均預測值
        final_prediction = np.mean(predictions)

        return {"predicted_spend_time_seconds": float(final_prediction)}

    except Exception as e:
        # 捕捉任何可能的錯誤
        raise HTTPException(status_code=400, detail=str(e))


result = predict(features)
print(result)

{'predicted_spend_time_seconds': 4764.80712890625}


In [43]:
# Test
features = Features(
    avg_temp=8.9,
    avg_rh=7,
    max_precip=0,
    distance=1713,
    elevation_range=526.2,
    elevation_change=-304.5,
    elevation_gain=19,
    elevation_loss=539.8,
    high_elevation=1,
    max_slope_percent=-74.2,
    max_slope_degrees=-36.55,
    slope_std_dev=10.57,
    slope_variance=111.64,
    max_slope_lat=24.412,
    max_slope_lon=121.309677,
    slope_neg15=67.27,
    slope_neg15_neg10=14.55,
    slope_neg10_neg5=3.64,
    slope_neg5_neg1=3.88,
    slope_neg1_1=3.64,
    slope_1_5=5.45,
    slope_5_10=1.82,
    slope_10_15=0,
    slope_over15=3,
    accumulated_time_seconds=30580,
    accumulated_distance=9813.28,
)

input_df = pd.DataFrame([features.model_dump()])

input_df

,avg_temp,avg_rh,max_precip,distance,elevation_range,elevation_change,elevation_gain,elevation_loss,high_elevation,max_slope_percent,...,slope_neg15_neg10,slope_neg10_neg5,slope_neg5_neg1,slope_neg1_1,slope_1_5,slope_5_10,slope_10_15,slope_over15,accumulated_time_seconds,accumulated_distance
0,8.9,7.0,0.0,1713.0,526.2,-304.5,19.0,539.8,1.0,-74.2,...,14.55,3.64,3.88,3.64,5.45,1.82,0.0,3.0,30580.0,9813.28


In [44]:
ordered_df = input_df[PIPELINE["features"]]
ordered_df

,accumulated_distance,accumulated_time_seconds,avg_rh,avg_temp,distance,elevation_change,elevation_gain,elevation_loss,elevation_range,high_elevation,...,slope_1_5,slope_5_10,slope_neg10_neg5,slope_neg15,slope_neg15_neg10,slope_neg1_1,slope_neg5_neg1,slope_over15,slope_std_dev,slope_variance
0,9813.28,30580.0,7.0,8.9,1713.0,-304.5,19.0,539.8,526.2,1.0,...,5.45,1.82,3.64,67.27,14.55,3.64,3.88,3.0,10.57,111.64


In [45]:
scaled_features = PIPELINE["scaler"].transform(ordered_df)
scaled_features

array([[0.45278079, 0.41637117, 0.07      , 0.44988345, 0.2379857 ,
        0.18361664, 0.01227787, 0.34882068, 0.35528697, 1.        ,
        0.        , 0.27829674, 0.98864679, 0.99998027, 0.45067932,
        0.        , 0.09772279, 0.03379759, 0.06759517, 0.87454498,
        0.37831513, 0.11829704, 0.06957145, 0.03900156, 0.43044292,
        0.26187427]])

In [46]:
predictions = [model.predict(scaled_features) for model in PIPELINE["models"]]

predictions

[array([4600.6475], dtype=float32),
 array([4558.96], dtype=float32),
 array([4934.995], dtype=float32),
 array([4689.9346], dtype=float32),
 array([5039.4985], dtype=float32)]

In [47]:
final_prediction = np.mean(predictions)
print(final_prediction)

4764.807
